In [ ]:
# !pip install -q ultralytics roboflow

In [ ]:
##Connect to Roboflow using Private API and download dataset
from roboflow import Roboflow

rf = Roboflow(api_key="roboflow_private_key")  #private API key

print("Successfully connected to Roboflow!")

WORKSPACE_ID = "roboflow-workspace_ID"
PROJECT_ID = "roboflow_project_ID"

workspace = rf.workspace(WORKSPACE_ID)
project = workspace.project(PROJECT_ID)
version = project.version(1)   #v1 = 1
dataset = version.download("yolov8")
print("Dataset downloaded successfully!")
print("Dataset location:", dataset.location)


In [ ]:
#Print Contents Downloaded
import os

print("\nDataset contents:")
for item in os.listdir(dataset.location):
    print(item)

#Read the contents of data.yaml
data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml, "r") as file:
    print(file.read())

In [ ]:
## Training the model

from ultralytics import YOLO
import os

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Train on your Roboflow dataset
results = model.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    name="ppe_detection"
)

print("Training completed!")

In [ ]:
from ultralytics import YOLO

best_model = YOLO("./best.pt")

print("Trained PPE model loaded successfully!")

#Predict on an image
results = best_model.predict(
    source="test_image.jpg", #image path
    conf=0.25,
    save=True  #saves the output image
)

print("Prediction completed!")

In [ ]:
##Test on your complete test dataset
from ultralytics import YOLO

best_model = YOLO("./best.pt")

metrics = best_model.val(
    data=DATA_YAML,
    split="test"
)

print("Test evaluation completed!")

In [ ]:
##Test using your webcam

from ultralytics import YOLO

best_model = YOLO("./best.pt")

best_model.predict(
    source=0,  #means your default webcam
    show=True,
    conf=0.25
)

In [ ]:
import glob

test_images = glob.glob('/workspace/PPE-DETECTION-1/test/images/*.jpg')[:10]

pred_results = model.predict(
    source=test_images,
    save=True,
    conf=0.25,
    project='/workspace/runs/detect', #Parent folder where results will be saved.
    name='sample_preds'	#Name of the prediction output folder.
)

print("Predictions saved successfully!")

In [ ]:
import os
import shutil

folder_name = "Anas Javed_2402900100039"
os.makedirs(folder_name, exist_ok=True)

# 1. Best weights copy karo
shutil.copy('/workspace/runs/detect/dgx_ppe_run/weights/best.pt', f'{folder_name}/best.pt')

# 2. Performance plots copy karo
shutil.copy('/workspace/runs/detect/dgx_ppe_run/results.png', f'{folder_name}/results.png')
shutil.copy('/workspace/runs/detect/dgx_ppe_run/confusion_matrix.png', f'{folder_name}/confusion_matrix.png')

# 3. Sample test predictions copy karo
if os.path.exists('/workspace/runs/detect/sample_preds'):
    shutil.copytree('/workspace/runs/detect/sample_preds', f'{folder_name}/sample_predictions', dirs_exist_ok=True)

# 4. ZIP archive create karo
!zip -r {folder_name}.zip {folder_name}

print(f"\nFinal submission ZIP ready: {folder_name}.zip")

In [ ]:
import shutil

folder_name = "Anas Javed_2402900100039"

# Pure Python se zip archive create karo
shutil.make_archive(folder_name, 'zip', folder_name)

print(f"Archive successfully created: {folder_name}.zip")